%matplotlib widget
%load_ext autoreload
%autoreload 2

import plot_polarsdf

In [1]:
%matplotlib widget
import pulsation_description as pls_d
import line_profile_description as lpd
import tomli_w
import polars as pl
import pulstar_py as pls_py
import os 
import matplotlib.pyplot as plt
import matplotlib.widgets as Slider

# Pulstar configuration
Here you specify the star you'll be modelling as well as the modes of pulsation

In [2]:
#Taken from a Simbad quick Query and from Teltings paper
mode=pls_d.Mode(l=2,m=1,
                rel_dr=0.024,
                k=0.03,frequency=5.38,
                phase_offset=0.0,
                rel_dtemp=2.62,
                phase_rel_dtemp=180.0,
                rel_dg=10.0,
                phase_rel_dg=34.0,
                rotation_effects = "PerturbativeCoriolis")
star_data=pls_d.StarData(mass=7.4,
                        radius=7.22,
                        effective_temperature=26000.0,
                        v_omega=54.0,
                        inclination_angle=60.0)
time_points=pls_d.TimePoints(Uniform=pls_d.UniformTime(start=0.0,end=6.65,step=0.01))
mesh=pls_d.Mesh(Sphere=pls_d.SphericalStar(theta_step=2.0,phi_step=4.0))

pulsconfig=pls_d.PulstarConfig(mode_data=[mode],star_data=star_data,time_points=time_points,mesh=mesh)
pulsconfig_dict=pulsconfig.model_dump(exclude_none=True)
puls_toml_string=tomli_w.dumps(pulsconfig_dict)

# Profile configuration
Here you specify the line profile variability you want to observe.

In [3]:
#Taken from a Simbad quick Query
wl_range=lpd.WavelengthRange(start=455.10,end=455.50,step=0.0033)
#path_to_grids="../profile/grids/"
path_to_grids = os.getenv("GRIDS")
grid1=lpd.IntensityGrid(Joris=lpd.JorisGrid(temperature=25000.0,log_gravity=3.5,filename="t25000g35.txt"),Nadya=None)
grid2=lpd.IntensityGrid(Joris=lpd.JorisGrid(temperature=25000.0,log_gravity=4.5,filename="t25000g45.txt"),Nadya=None)
grid3=lpd.IntensityGrid(Joris=lpd.JorisGrid(temperature=27000.0,log_gravity=3.5,filename="t27000g35.txt"),Nadya=None)
grid4=lpd.IntensityGrid(Joris=lpd.JorisGrid(temperature=27000.0,log_gravity=4.5,filename="t27000g45.txt"),Nadya=None)
prof_config=lpd.ProfileConfig(max_velocity=1.0e2,path_to_grids=path_to_grids,wavelength_range=wl_range,intensity_grids=[grid1,grid2,grid3,grid4])

prof_config_dict=prof_config.model_dump(exclude_none=True)

prof_toml_string=tomli_w.dumps(prof_config_dict)


# First run

In [ ]:
pulse_df = pls_py.pulstar(puls_toml_string)

--------------------
--------------------
--------------------
|PULSTARust launched|
--------------------

 +-- Computing surface data for time point number 0 with time stamp 0.000.

 +-- Computing surface data for time point number 1 with time stamp 0.010.

 +-- Computing surface data for time point number 2 with time stamp 0.020.

 +-- Computing surface data for time point number 3 with time stamp 0.030.

 +-- Computing surface data for time point number 4 with time stamp 0.040.

 +-- Computing surface data for time point number 5 with time stamp 0.050.

 +-- Computing surface data for time point number 6 with time stamp 0.060.

 +-- Computing surface data for time point number 7 with time stamp 0.070.

 +-- Computing surface data for time point number 8 with time stamp 0.080.

 +-- Computing surface data for time point number 9 with time stamp 0.090.

 +-- Computing surface data for time point number 10 with time stamp 0.100.

 +-- Computing surface data for time point number 11 wit

In [ ]:
df=pls_py.propulse(prof_toml_string,puls_toml_string)
#pulse_df.write_parquet("star_model2m1.parquet")
#print(df.head(5))

In [ ]:
grouped=df.lazy().group_by("time").agg(pl.col("wave length"),pl.col("normalized flux"))
#print(grouped.clone().sort("time").collect().head(5))

In [ ]:
grouped_df=grouped.collect()

In [ ]:
time = grouped_df.get_column("time").sort()

In [ ]:
single_df = df.lazy().filter(pl.col("time").eq(time[3])).collect()

In [ ]:
wavelength = list(single_df['wave length'])
intensity = list(single_df['normalized flux'])

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(wavelength,intensity)
plt.xlabel("wavelength (nm)")
plt.ylabel("flux")
plt.title("Plot of normalized flux versus lambda")
plt.grid(True)
# Save the plot as a JPG image

local_filename = (f"Normalized_intensity4")
plt.savefig(local_filename, format='jpeg', dpi=300) # dpi for higher resolution
print(f"Plot saved successfully as '{local_filename}'")
        
plt.show()

In [ ]:
df.write_parquet(file="wavelenght.parquet")

In [ ]:
from matplotlib.widgets import Slider


In [ ]:
def plot_df(df:pl.DataFrame):
    grouped=df.lazy().group_by("time").agg(pl.col("wave length"),pl.col("normalized flux"))
    grouped_df=grouped.sort("time").collect()
    time_list=grouped_df.get_column("time").sort()


    # Setup plot
    fig, ax = plt.subplots(figsize=(10,7))
    plt.subplots_adjust(bottom=0.25)
    
    #single_df=(grouped_df.lazy()
    #.filter(pl.col("time")
    #.eq(time_list[0]))
    #.collect())
    single_df=df.filter(pl.col("time") == time_list[0])
    wavelength = list(single_df["wave length"])
    intensity = list(single_df["normalized flux"])

    line, = ax.plot(single_df["wave length"].to_numpy(),single_df["normalized flux"].to_numpy(),lw=2,color='crimson')
    #line = ax.plot(single_df)
    ax.set_xlabel("wavelength")
    ax.set_ylabel("normalized flux")
    ax.set_title(f"time point {time_list[0]:.3f}")
    ax.grid(True)

    #Create Slider Ax
    ax_slider = plt.axes([0.2,0.1,0.6,0.03])
    time_slider = Slider(
        ax=ax_slider,
        label='Time',
        valmin=min(time_list),
        valmax=max(time_list),
        valinit=time_list[0],
        valstep=time_list)
    
    #update function
    def update(val):
        selected_time = time_slider.val
        #new_df=(grouped_df.lazy()
        #.filter(pl.col("time")
        #.eq(selected_time))
        #.collect())
        new_df=df.filter(pl.col("time")==val)
        new_wavelength = list(single_df['wave length'])
        new_intensity = list(single_df['normalized flux'])
        #line.set_xdata(new_wavelength)
        #line.set_ydata(new_intensity)
        line.set_xdata(new_df["wave length"].to_numpy())
        line.set_ydata(new_df["normalized flux"].to_numpy())
        ax.set_title(f"time point {selected_time:.3f}")
        fig.canvas.draw_idle()
    
    time_slider.on_changed(update)

    plt.show()






In [ ]:

plot_df(df)